In [82]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

In [83]:
load_dotenv()

True

In [84]:
api_key = os.getenv("GROQ_API_KEY")

In [ ]:
llm = ChatGroq(
    model='qwen/qwen3.8-27b',
    temperature=0.2,
    max_tokens=1000
)

In [86]:
code = """
def divide(a, b):
    return a / b

result = divide(10, 0)
print(result)
"""

print(code)


def divide(a, b):
    return a / b

result = divide(10, 0)
print(result)



In [87]:
from langchain_core.prompts import ChatPromptTemplate

In [88]:
prompt = ChatPromptTemplate.from_template("""
You are an expert software engineer and professional code reviewer.

Your task is to review the submitted source code.

Programming Language: {language}

Source Code:
{code}

Follow these instructions carefully:

1. Analyze the code using the specified programming language.
2. Identify bugs, runtime errors and logical errors.
3. Explain why each issue occurs.
4. Identify potential security vulnerabilities.
5. Identify performance issues.
6. Suggest code quality improvements.
7. Provide corrected code.

IMPORTANT RULES:
- Never change the original programming language.
- Do not convert the code into Python or any other language.
- The corrected code must use {language}.
- Preserve the original functionality.
- If no issue is found, clearly mention that.
- Do not invent errors that are not supported by the code.

""")


In [89]:
from langchain_core.output_parsers import StrOutputParser

In [90]:
parser = StrOutputParser()

In [91]:
markdown_instructions = (
    'Use these Markdown headings: Bugs Identified; Explanation of Issues; Security Analysis; '
    'Performance Analysis; Improvement Suggestions; Corrected Code.'
)
output = prompt.partial(format_instructions=markdown_instructions) | llm | parser

In [92]:
output.invoke({"language":"python","code":code})

'# Code Review: Python Division Function\n\n## 1. Analysis\n\nThe code defines a simple `divide` function that performs division and immediately calls it with arguments `(10, 0)`.\n\n## 2. Bugs, Runtime Errors, and Logical Errors\n\n### Critical: ZeroDivisionError (Runtime Error)\n\n```python\nresult = divide(10, 0)\n```\n\n**Why this occurs:** In Python, dividing any number by zero raises a `ZeroDivisionError`. The function `divide(a, b)` performs `a / b` without any check for whether `b` is zero. When called with `b=0`, the program will crash with:\n\n```\nZeroDivisionError: division by zero\n```\n\nThis is not a logical error in the function\'s design per se, but the **caller** invokes the function with an invalid input, and the function provides no protection against it. The code as written will **never reach the `print(result)` line**.\n\n## 3. Security Vulnerabilities\n\nNo direct security vulnerabilities are present in this snippet. However, if this function were exposed in a we

In [93]:
java_code = """
public class Main {
    public static void main(String[] args) {
        int[] numbers = {10, 20, 30};

        System.out.println(numbers[5]);
    }
}
"""

In [105]:
output.invoke({"language":"java","code":java_code})

'# Code Review: Java `Main.java`\n\n## 1. Analysis\n\nThe code defines a simple Java program that creates an integer array of size 3 and attempts to access an element at index 5.\n\n## 2. Bugs, Runtime Errors, and Logical Errors\n\n### Critical Bug: `ArrayIndexOutOfBoundsException`\n\n```java\nint[] numbers = {10, 20, 30};  // Array has indices 0, 1, 2\nSystem.out.println(numbers[5]); // Index 5 is out of bounds\n```\n\n**Why this occurs:** The array `numbers` has a length of 3, meaning valid indices are `0`, `1`, and `2`. Accessing `numbers[5]` exceeds the valid range. In Java, this will throw an `ArrayIndexOutOfBoundsException` at runtime, causing the program to crash.\n\nThis is the **only** bug in the code. There are no other logical errors, security vulnerabilities, or performance issues present.\n\n## 3. Security Vulnerabilities\n\nNone identified. The code does not handle external input, perform file I/O, network operations, or any other security-sensitive operations.\n\n## 4. P

In [95]:
from pydantic import Field,BaseModel
from typing import List

In [96]:
class CodeReview(BaseModel):
    bugs: list[str] = Field(
        description="List of bugs and errors identified in the code"
    )
    security_issues: list[str] = Field(
        description="List of potential security vulnerabilities"
    )
    performance_issues: list[str] = Field(
        description="List of performance-related issues"
    )
    improvement_suggestions: list[str] = Field(
        description="Suggestions to improve code quality"
    )
    corrected_code: str = Field(
        description="Corrected source code in the original programming language"
    )

In [101]:
code_review_chain = prompt | llm.with_structured_output(CodeReview)

In [102]:
java_code = """
public class Main {
    public static void main(String[] args) {
        int[] numbers = {10, 20, 30};

        System.out.println(numbers[5]);
    }
}
"""

In [103]:
code_review_chain.invoke({"language":"java","code":java_code})

CodeReview(bugs=['ArrayIndexOutOfBoundsException: The code attempts to access index 5 of an array with only 3 elements (indices 0, 1, 2). This will throw a java.lang.ArrayIndexOutOfBoundsException at runtime.'], security_issues=['No security vulnerabilities identified in this code.'], performance_issues=['No significant performance issues identified in this code.'], improvement_suggestions=['Add bounds checking before accessing array elements to prevent runtime exceptions.', 'Consider using a loop or enhanced for-loop to iterate over array elements safely.', 'Add error handling (try-catch) if array access with dynamic indices is necessary.', 'Use meaningful variable names and add comments for clarity.'], corrected_code='public class Main {\n    public static void main(String[] args) {\n        int[] numbers = {10, 20, 30};\n\n        // Check if the index is within bounds before accessing\n        int index = 5;\n        if (index >= 0 && index < numbers.length) {\n            System.o

In [106]:
def review_code(language, code):

    try:
        result = code_review_chain.invoke({
            "language": language,
            "code": code
        })

        return result

    except Exception as e:
        print(f"Code review failed: {e}")
        return None

In [107]:
code = """
def divide(a, b):
    return a / b

print(divide(10, 0))
"""

result = review_code("Python", code)

if result:
    print(result.model_dump())

Code review failed: Error code: 400 - {'error': {'message': "tool call validation failed: attempted to call tool 'Code Review' which was not in request.tools", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<tool_call>\n<function=Code Review>\n<parameter=bugs>\n["Division by zero: The function `divide` does not handle the case where `b` is 0. When called as `divide(10, 0)`, it raises a `ZeroDivisionError` at runtime."]\n</parameter>\n<parameter=security_issues>\n["No direct security vulnerabilities identified in this snippet. However, if this function were exposed in a web service or API without input validation, it could be exploited to cause denial of service via unhandled exceptions."]\n</parameter>\n<parameter=performance_issues>\n["No significant performance issues identified. The function is a simple arithmetic operation."]\n</parameter>\n<parameter=improvement_suggestions>\n["Add input validation to check if the divisor is zero before performin